In [ ]:
import numpy as np
import pandas as pd
import os 
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 64
N_EPOCHS = 100
LEARNING_RATE = 0.001



In [ ]:
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
DATA_DIR = '../data/processed/'
X_train = np.load(DATA_DIR + 'X_train.npy')
X_val   = np.load(DATA_DIR + 'X_val.npy')
X_test  = np.load(DATA_DIR + 'X_test.npy')
y_train = np.load(DATA_DIR + 'y_train.npy')
y_val   = np.load(DATA_DIR + 'y_val.npy')
y_test  = np.load(DATA_DIR + 'y_test.npy')

In [ ]:
class HousingData(Dataset):
    def __init__(self,X,y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class PolynomialRegressionModel(nn.Module):
    def __init__(self, n_features):
        super(PolynomialRegressionModel, self).__init__()
        self.linear = nn.Linear(n_features, 1)
    def forward(self, x):
        return self.linear(x)

In [ ]:
for M in [1, 2, 3, 4, 5]:
    poly = PolynomialFeatures(degree=M, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_val_poly   = poly.transform(X_val)
    X_test_poly  = poly.transform(X_test)

    poly_scaler  = StandardScaler()                            
    X_train_poly = poly_scaler.fit_transform(X_train_poly)    
    X_val_poly   = poly_scaler.transform(X_val_poly)          
    X_test_poly  = poly_scaler.transform(X_test_poly)  

    X_train_t = torch.tensor(X_train_poly, dtype = torch.float32)
    X_val_t = torch.tensor(X_val_poly, dtype = torch.float32)
    X_test_t = torch.tensor(X_test_poly, dtype = torch.float32)

    y_train_t = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)
    
    training_dataset = HousingData(X_train_t, y_train_t)
    valiation_dataset = HousingData(X_val_t, y_val_t)
    test_dataset = HousingData(X_test_t, y_test_t)
    
    train_loader = DataLoader(training_dataset, shuffle= True, num_workers=0, batch_size=BATCH_SIZE)
    val_loader = DataLoader(valiation_dataset, batch_size= BATCH_SIZE, shuffle= False, num_workers = 0)
    test_loader = DataLoader(test_dataset, batch_size= BATCH_SIZE, shuffle = False, num_workers= 0)

    N_FEATURES = X_train_poly.shape[1]
    model = PolynomialRegressionModel(n_features=N_FEATURES)
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr =LEARNING_RATE)
    train_losses = []
    val_losses = []
    
    for epoch in range(N_EPOCHS):
        model.train()
        epoch_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss/len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        epoch_val_loss = 0.0

        with torch.no_grad():
            
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                val_predictions = model(X_batch)
                val_loss = criterion(val_predictions, y_batch)
                epoch_val_loss += val_loss.item()

            avg_val_loss = epoch_val_loss / len(val_loader)
            val_losses.append(avg_val_loss)

            if (epoch + 1) % 10 == 0:
                print(f'Epoch [{epoch+1:3d}/{N_EPOCHS}]  '
                  f'Train MSE: {avg_train_loss:.4f}  '
                  f'Val MSE:   {avg_val_loss:.4f}')

    print(f'for degree: {M}')
    print('trainig complete')

    plt.figure(figsize=(10, 5))
    epochs_range = range(1, N_EPOCHS + 1)
    plt.plot(epochs_range, train_losses, label='Train MSE', color='steelblue',  linewidth=2)
    plt.plot(epochs_range, val_losses,   label='Val MSE',   color='coral',      linewidth=2, linestyle='--')

    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Learning Curve — Polynomial Regression (California Housing)')
    plt.legend()
    plt.grid(True, alpha=0.3)  
    plt.tight_layout()

    plt.savefig(f'../results/plots/poly_deg{M}_learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()  

    model.eval()
    all_preds = []
    all_true  = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch)

            all_preds.append(preds.cpu().numpy().flatten())
            all_true.append(y_batch.cpu().numpy().flatten())

    y_pred_val = np.concatenate(all_preds)
    y_true_val = np.concatenate(all_true)

    mse  = np.mean((y_pred_val - y_true_val) ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(y_pred_val - y_true_val))
    print(f' === test Set Metrics for degree {M} ===')
    print(f'  MSE:  {mse:.4f}  (units: $100k squared)')
    print(f'  RMSE: {rmse:.4f}  (units: $100k  → multiply by 100,000 for dollars)')
    print(f'  MAE:  {mae:.4f}  (units: $100k)')
    print()
    print(f'  In plain English: predictions are off by ~${rmse*100000:,.0f} on average (RMSE)') 

    os.makedirs('../results/models', exist_ok = True)
    model_path = f'../results/models/poly_deg{M}_california.pth'
    torch.save(model.state_dict(), model_path)

    
    log_path = '../results/logs/experiment_log.csv'
    os.makedirs('../results/logs', exist_ok=True)

    result_row = {
        'experiment':  'polynomial_regression',
        'dataset':     'california_housing',
        'model_type':  'polynomial',
        'degree_M':    M,
        'lambda':      None,
        'n_epochs':    N_EPOCHS,
        'batch_size':  BATCH_SIZE,
        'lr':          LEARNING_RATE,
        'train_mse':   round(train_losses[-1], 4),
        'val_mse':     round(val_losses[-1],   4),
        'test_mse':    round(mse,              4),
        'test_rmse':   round(rmse,             4),
        'test_mae':    round(mae,              4),
    }

    result_df = pd.DataFrame([result_row])

    if os.path.exists(log_path):
        result_df.to_csv(log_path, mode='a', header=False, index=False)
    else:
        result_df.to_csv(log_path, mode='w', header=True,  index=False)

    print(f'Logged M={M} results to {log_path}')
    print()
        